# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle
import numpy as np
import tqdm
import os
import glob
import multiprocessing as mp

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer


# get patient traj, keep only snomed and build vocabs

In [15]:
df_traj = pl.read_parquet(config.TimelineData().all_patient_traj).filter(pl.col("mapping_type") == "SNOMED")
df_gender = pl.read_parquet(config.TimelineData().patient_gender).unique()

In [20]:
df_traj.select("mapped_code", "mapped_label").unique().write_parquet(config.TimelineVocab().snomed_vocabs_original)

In [22]:
len(pl.read_parquet(config.TimelineVocab().snomed_vocabs_original)) + len(config.TimelineVocab().gender_vocab)

6004

# apply tokenizer of these data vocabs with all configs of all baseline and greedy tree

In [4]:
df_mapped = pl.read_parquet(config.TimelineVocab().snomed_vocabs_original)
mapped_ids = config.TimelineVocab().gender_vocab + df_mapped["mapped_code"].unique().to_list()

lambdas_list = np.round(np.arange(0, 1.2, 0.2), 1)
Ks = np.arange(100, 6100, 100)
rnd_iters = config.TokenizerParam().rnd_iters
D = config.TokenizerParam().max_dist_candidate
K_TARGET = int(Ks.max())  # rank exactly as many candidates as the eval sweep will ever slice at

print(f"mapped_ids: {len(mapped_ids):,}   D: {D}   max(Ks): {Ks.max():,}   K_TARGET: {K_TARGET:,}")

mapped_ids: 6,004   D: 3   max(Ks): 6,000   K_TARGET: 6,000


# build the case-study candidate graph (D-hop ego-graph union over this M)

In [5]:
import networkx as nx

import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct
import src.graph_tokenizer_gd_tree_dev.classical_selectors as classical_selectors

df_relations = pl.read_parquet(config.BasicConfig().relation_path)
df_relations = df_relations.filter(~pl.col("dst.id").is_in(config.TokenizerParam().exclude_cpt))

whole_graph = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
combined_subgraphs = graph_fct.get_combined_subgraphs_from_nodes(whole_graph, mapped_ids, max_distance=D)

os.makedirs(config.IAProcessedGraph().path, exist_ok=True)
with open(config.IAProcessedGraph().combined_subgraphs, "wb") as f:
    pickle.dump(combined_subgraphs, f)

# id_to_label is a global concept_id -> label map, not M-specific -- reuse the main pipeline's
with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

print(f"combined_subgraphs: {combined_subgraphs.number_of_nodes():,} nodes, {combined_subgraphs.number_of_edges():,} edges")

combined_subgraphs: 18,119 nodes, 54,595 edges


# greedy tree selection across all lambdas

In [6]:
os.makedirs(config.IACandidateLists().path_greedy_tree, exist_ok=True)

for lam in tqdm.tqdm(lambdas_list):
    selector = tokenizer.LazyGreedyTokenSelector(combined_subgraphs, mapped_ids, D=D, lam=lam)
    history = selector.select(k=K_TARGET, n_jobs=-1)  # [(token, marginal_gain, cumulative_score), ...]
    df_history = (
        pl.DataFrame(history, schema=["token", "gain", "cumulative_score"], orient="row")
        .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
        .with_row_index()
    )
    df_history.write_parquet(f"{config.IACandidateLists().path_greedy_tree}{lam}.parquet")

  0%|          | 0/6 [00:00<?, ?it/s]

  seeding candidates: 0/18119  (12 workers)
  seeding candidates: 5000/18119  (12 workers)
  seeding candidates: 10000/18119  (12 workers)
  seeding candidates: 15000/18119  (12 workers)
  seeding candidates: 18119/18119  (12 workers)
[   1/6000] token='10050004'      gain=0.00017  score=0.00017  opt<=0.00026  (re-evals=1, queue=18118)
[   2/6000] token='10083006'      gain=0.00017  score=0.00033  opt<=0.00053  (re-evals=1, queue=18117)
[   3/6000] token='10085004'      gain=0.00017  score=0.00050  opt<=0.00079  (re-evals=1, queue=18116)
[   4/6000] token='101401000119103' gain=0.00017  score=0.00067  opt<=0.00105  (re-evals=1, queue=18115)
[   5/6000] token='1023001'       gain=0.00017  score=0.00083  opt<=0.00132  (re-evals=1, queue=18114)
[   6/6000] token='102449007'     gain=0.00017  score=0.00100  opt<=0.00158  (re-evals=1, queue=18113)
[   7/6000] token='102478008+389087006' gain=0.00017  score=0.00117  opt<=0.00184  (re-evals=1, queue=18112)
[   8/6000] token='102486008'     ga

 17%|█▋        | 1/6 [00:26<02:14, 26.87s/it]

  seeding candidates: 0/18119  (12 workers)
  seeding candidates: 5000/18119  (12 workers)
  seeding candidates: 10000/18119  (12 workers)
  seeding candidates: 15000/18119  (12 workers)
  seeding candidates: 18119/18119  (12 workers)
[   1/6000] token='129276002'     gain=0.00254  score=0.00254  opt<=0.00402  (re-evals=1, queue=18118)
[   2/6000] token='129304002'     gain=0.00121  score=0.00375  opt<=0.00594  (re-evals=1, queue=18117)
[   3/6000] token='410604004'     gain=0.00105  score=0.00480  opt<=0.00760  (re-evals=1, queue=18116)
[   4/6000] token='409774005'     gain=0.00103  score=0.00583  opt<=0.00923  (re-evals=1, queue=18115)
[   5/6000] token='129445006'     gain=0.00097  score=0.00680  opt<=0.01076  (re-evals=1, queue=18114)
[   6/6000] token='404684003'     gain=0.00091  score=0.00772  opt<=0.01221  (re-evals=1, queue=18113)
[   7/6000] token='768681000'     gain=0.00087  score=0.00859  opt<=0.01359  (re-evals=1, queue=18112)
[   8/6000] token='410942007'     gain=0.000

 33%|███▎      | 2/6 [00:44<01:25, 21.37s/it]

...

 67%|██████▋   | 4/6 [01:32<00:49, 24.53s/it]

  seeding candidates: 0/18119  (12 workers)
  seeding candidates: 5000/18119  (12 workers)
  seeding candidates: 10000/18119  (12 workers)
  seeding candidates: 15000/18119  (12 workers)
  seeding candidates: 18119/18119  (12 workers)
[   1/6000] token='129264002'     gain=0.02164  score=0.02164  opt<=0.03424  (re-evals=1, queue=18118)
[   2/6000] token='726711005'     gain=0.01498  score=0.03662  opt<=0.05793  (re-evals=2, queue=18117)
[   3/6000] token='410942007'     gain=0.01426  score=0.05088  opt<=0.08049  (re-evals=1, queue=18116)
[   4/6000] token='404684003'     gain=0.01411  score=0.06499  opt<=0.10281  (re-evals=1, queue=18115)
[   5/6000] token='49755003'      gain=0.01282  score=0.07781  opt<=0.12310  (re-evals=1, queue=18114)
[   6/6000] token='91689009'      gain=0.01137  score=0.08918  opt<=0.14108  (re-evals=2, queue=18113)
[   7/6000] token='182353008'     gain=0.01135  score=0.10054  opt<=0.15905  (re-evals=1, queue=18112)
[   8/6000] token='129276002'     gain=0.010

 83%|████████▎ | 5/6 [02:07<00:28, 28.10s/it]

  seeding candidates: 0/18119  (12 workers)
  seeding candidates: 5000/18119  (12 workers)
  seeding candidates: 10000/18119  (12 workers)
  seeding candidates: 15000/18119  (12 workers)
  seeding candidates: 18119/18119  (12 workers)
[   1/6000] token='129264002'     gain=0.04080  score=0.04080  opt<=0.06455  (re-evals=1, queue=18118)
[   2/6000] token='726711005'     gain=0.02679  score=0.06759  opt<=0.10693  (re-evals=1, queue=18117)
[   3/6000] token='410942007'     gain=0.02426  score=0.09185  opt<=0.14531  (re-evals=1, queue=18116)
[   4/6000] token='404684003'     gain=0.02410  score=0.11595  opt<=0.18344  (re-evals=1, queue=18115)
[   5/6000] token='49755003'      gain=0.02278  score=0.13874  opt<=0.21948  (re-evals=1, queue=18114)
[   6/6000] token='91689009'      gain=0.02029  score=0.15903  opt<=0.25158  (re-evals=3, queue=18113)
[   7/6000] token='182353008'     gain=0.01966  score=0.17869  opt<=0.28268  (re-evals=2, queue=18112)
[   8/6000] token='71388002'      gain=0.013

100%|██████████| 6/6 [02:46<00:00, 27.69s/it]


# D-hop distance tables (all-relation and IS_A-only) -- shared by highest_degree, most_children, discrete_set_cover

In [7]:
# all relationship distances in the combined subgraph
all_pairs = nx.all_pairs_shortest_path_length(combined_subgraphs)
distance_df = pl.DataFrame(
    [
        {"src_id": src, "dst_id": dst, "distance": dist}
        for src, lengths in all_pairs
        for dst, dist in lengths.items()
        if src != dst
    ]
)
self_distance_df = pl.DataFrame([{"src_id": n, "dst_id": n, "distance": 0} for n in combined_subgraphs.nodes()])
distance_df = pl.concat([distance_df, self_distance_df])
distance_df.write_parquet(config.IAProcessedGraph().all_rel_distance_subgraph)

# is_a only distances in the combined subgraph
is_a_edges = [
    (u, v, k) for u, v, k, attrs in combined_subgraphs.edges(keys=True, data=True)
    if attrs.get("relation") == "IS_A"
]
is_a_subgraph = nx.MultiDiGraph(combined_subgraphs.edge_subgraph(is_a_edges))  # materialize, don't keep it a view

all_pairs = nx.all_pairs_shortest_path_length(is_a_subgraph)
is_a_distance_df = pl.DataFrame(
    [
        {"src_id": src, "dst_id": dst, "distance": dist}
        for src, lengths in all_pairs
        for dst, dist in lengths.items()
        if src != dst
    ]
)
is_a_self_distance_df = pl.DataFrame([{"src_id": n, "dst_id": n, "distance": 0} for n in combined_subgraphs.nodes()])
is_a_distance_df = pl.concat([is_a_distance_df, is_a_self_distance_df])
is_a_distance_df.write_parquet(config.IAProcessedGraph().is_a_distance_subgraph)

# baselines: highest_degree / highest_degree_dist_1 / most_children

In [10]:
(pl.read_parquet(config.IAProcessedGraph().all_rel_distance_subgraph)
 .filter(pl.col("distance") <= D)
 .group_by("dst_id")
 .agg(pl.col("src_id").n_unique().alias("num_in_edges"))
 .sort(by="num_in_edges", descending=True)
 .with_row_index()
 .rename({"dst_id": "token"})
 .write_parquet(config.IACandidateLists().highest_degree))

MAX_DIST = 1
(pl.read_parquet(config.IAProcessedGraph().all_rel_distance_subgraph)
 .filter(pl.col("distance") <= MAX_DIST)
 .group_by("dst_id")
 .agg(pl.col("src_id").n_unique().alias("num_in_edges"))
 .sort(by="num_in_edges", descending=True)
 .with_row_index()
 .rename({"dst_id": "token"})
 .write_parquet(config.IACandidateLists().highest_degree_dist_1))

(pl.read_parquet(config.IAProcessedGraph().is_a_distance_subgraph)
 .filter(pl.col("distance") <= D)
 .group_by("dst_id")
 .agg(pl.col("src_id").n_unique().alias("num_in_edges"))
 .sort(by="num_in_edges", descending=True)
 .with_row_index()
 .rename({"dst_id": "token"})
 .write_parquet(config.IACandidateLists().most_children))

# baselines: pagerank / personalized_pagerank / closeness_centrality / eigenvector_centrality

In [11]:
def to_ranked_df(scores: dict, score_col: str) -> pl.DataFrame:
    return (
        pl.DataFrame({"token": list(scores.keys()), score_col: list(scores.values())})
        .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
        .sort(score_col, descending=True)
        .with_row_index()
    )

pagerank = nx.pagerank(combined_subgraphs)
to_ranked_df(pagerank, "pagerank").write_parquet(config.IACandidateLists().pagerank)

personalized_pagerank = nx.pagerank(
    combined_subgraphs,
    personalization={m: 1 for m in mapped_ids if m in combined_subgraphs},
)
to_ranked_df(personalized_pagerank, "personalized_pagerank").write_parquet(config.IACandidateLists().personalized_pagerank)

G_simple = nx.DiGraph(combined_subgraphs)  # multi-edges collapsed, shared by both cells below

closeness_centrality = nx.closeness_centrality(G_simple)
to_ranked_df(closeness_centrality, "closeness_centrality").write_parquet(config.IACandidateLists().closeness_centrality)

eigenvector_centrality = nx.eigenvector_centrality(G_simple, max_iter=500)
to_ranked_df(eigenvector_centrality, "eigenvector_centrality").write_parquet(config.IACandidateLists().eigenvector_centrality)

# baseline: discrete (hard) greedy set-cover -- the central soft-vs-hard ablation

In [12]:
distance_df = pl.read_parquet(config.IAProcessedGraph().all_rel_distance_subgraph)
set_cover_selector = classical_selectors.HardGreedySetCoverSelector(distance_df, mapped_ids, D)
history_set_cover = set_cover_selector.select(k=K_TARGET, verbose=True, progress_every=2000)

(pl.DataFrame(history_set_cover, schema=["token", "gain", "cumulative_score"], orient="row")
 .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
 .with_row_index()
 .write_parquet(config.IACandidateLists().discrete_set_cover))

print(f"ranked {len(history_set_cover):,} candidates "
      f"({len(set_cover_selector.covers):,} cover >=1 mapped concept, "
      f"{len(set_cover_selector._leftover):,} zero-coverage padded)")

[  2000/6000] token='128050000'     gain=   0  covered=5816/6004 (96.869%)  (re-evals=1)
[  4000/6000] token='245125006'     gain=   0  covered=5816/6004 (96.869%)  (re-evals=1)
[  6000/6000] token='304120007'     gain=   0  covered=5816/6004 (96.869%)  (re-evals=1)
ranked 6,000 candidates (13,822 cover >=1 mapped concept, 0 zero-coverage padded)


# commpare all configs and methods